In [ ]:
# loading important packages
import numpy as np 
import pandas as pd 
import sys
!{sys.executable} -m pip install splink
import splink.comparison_library as cl
from splink import comparison_level_library as cll
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

In [ ]:
# Load in mentions data
mentions_new = pd.read_csv('mentions/mentions.csv')
# Filter mentions
mentions = mentions_new[
    (mentions_new['source'] == 'ALB_CN_1870')|
     (mentions_new['source'] == 'ALB_CN_1880')  
    ]
# Rename the mentions_id column to unique_id
mentions = mentions.rename(columns={"mention_id": "unique_id"})
# Remove duplicates
mentions = mentions.drop_duplicates(subset=['unique_id'])
# Drop unused cols.
mentions = mentions.drop(columns=['source', 'confidence', 'race', 'occupation', 'created','full_name','first_name','last_name'])
# Normalize all middle names to be uppercase
mentions['middle_name'] = mentions['middle_name'].str.upper()
# Drop more cols.
mentions = mentions.drop(columns=['maiden_name','death_year','legal_status','is_enslaver','location_id'])
# Separate mentions data into 1870 and 1880 records
mentions_1870 = mentions[mentions['source_year'] == 1870]
mentions_1880 = mentions[mentions['source_year'] == 1880]

In [ ]:
#creating custom levels for birth_year
birth_year_comparison = cl.CustomComparison(
    output_column_name = "birth_year",
    comparison_levels = [
        cll.NullLevel("birth_year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) = 0",
            label_for_charts = "Exact Birth Year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 1",
            label_for_charts = "Within 1 Year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 2",
            label_for_charts = "Within 2 Years"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 5",
            label_for_charts = "Within 5 Years"),
        cll.ElseLevel()
          
    ]
)
# the ordering of the following matters so john and joan would fall into "within 1 letter" rather than "same inital letter"
first_name_comparison = cl.CustomComparison(
    output_column_name = "norm_first_name",
    comparison_levels = [
        cll.NullLevel("norm_first_name"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 0",
            label_for_charts = "Exact First Name"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 1",
            label_for_charts = "Within 1 Letter"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 2",
            label_for_charts = "Within 2 Letters"),
        cll.CustomLevel(
            "substr(norm_first_name_l, 1, 1) = substr(norm_first_name_r, 1, 1)",
            label_for_charts = "Same First Initial"),
        cll.ElseLevel()
          
    ]
)

In [ ]:
# the following code simply specifies the linkage model
# settings = SettingsCreator(
#     link_type="link_only",
#     comparisons=[                               # everything in the comparisons is what is used to determine the match score
#         cl.ExactMatch("gender"),   
#         cl.ExactMatch("norm_race"),
#         cl.NameComparison("nysiis_last_name"), # there are 5 default comparison levels : exact match, null match, and 3 different levels based on the jaro-winkler similarity
#         birth_year_comparison,            # 6 custom comparison levels
#         first_name_comparison             # 5 custom comparison level
#     ],
#     blocking_rules_to_generate_predictions=[ #  will only look at records in which the race gender are the same 
#         block_on("gender","norm_race") # potential to insert multi-pass blocking HERE; running multiple iterations; also block on absolute value of birth year 
#     ],
#     retain_intermediate_calculation_columns=True,   # the runtime for the computations will be longer but will output more information so that i can understand the calculations
# )
settings = {
    "link_type": "link_only",

    "blocking_rules_to_generate_predictions": [
        "l.norm_first_name = r.norm_first_name and l.nysiis_last_name = r.nysiis_last_name",
        "l.norm_race = r.norm_race and l.gender = r.gender and abs(l.birth_year - r.birth_year) <= 5",
    ],

    "comparisons": [
    birth_year_comparison,
    cl.ExactMatch("norm_race"),
    cl.ExactMatch("gender"),
    first_name_comparison,
]
}
linker = Linker([mentions_1870, mentions_1880], settings, db_api=DuckDBAPI())

In [ ]:
df_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_predictions.as_pandas_dataframe(limit=5)